<h1>Vectorization</h1>

In [1]:
from torchtext.data.utils import get_tokenizer

sample1 = 'We are learning AI'
sample2 = 'AI is a CS topic'

# Define tokenizer function
tokenizer = get_tokenizer('basic_english')
sample1_tokens = tokenizer(sample1)
sample2_tokens = tokenizer(sample2)

print(sample1_tokens)
print(sample2_tokens)

['we', 'are', 'learning', 'ai']
['ai', 'is', 'a', 'cs', 'topic']


In [2]:
from torchtext.vocab import build_vocab_from_iterator
# Create a function to yield list of tokens
data = [sample1, sample2]
def yield_tokens(examples):
    for text in examples:
        yield tokenizer(text)

# Create vocabulary

vocab_size = 8
vocab = build_vocab_from_iterator(
    yield_tokens(data),
    max_tokens = vocab_size,
    specials = ['<unk>', '<pad>']
)
vocab.set_default_index(vocab['<unk>'])

vocab.get_stoi()

{'learning': 7,
 'is': 6,
 'are': 4,
 'a': 3,
 'cs': 5,
 'ai': 2,
 '<pad>': 1,
 '<unk>': 0}

In [4]:
tokens = tokenizer(sample1)
print(tokens)

sample_tokens = [vocab[token] for token in tokens]
print(sample_tokens)

['we', 'are', 'learning', 'ai']
[0, 4, 7, 2]


In [6]:
import torch 
def vectorize(text, vocab, sequence_length):
    tokens = tokenizer(text)
    tokens = [vocab[token] for token in tokens]

    if len(tokens) < sequence_length:
        tokens = tokens + [vocab['<padd>']] * (sequence_length - len(tokens))
    else:
        tokens = tokens[:sequence_length]
    return torch.tensor(tokens, dtype = torch.long)


sequence_length = 6
vectorized_sample1 = vectorize(sample1, vocab, sequence_length)

vectorized_sample2 = vectorize(sample2, vocab, sequence_length)

print(f'Vectorized Sample 1: {vectorized_sample1}')
print(f'Vectorized Sample 2: {vectorized_sample2}')

Vectorized Sample 1: tensor([0, 4, 7, 2, 0, 0])
Vectorized Sample 2: tensor([2, 6, 3, 5, 0, 0])


<h1>Embedding</h1>

In [16]:
import torch 
from torch import nn 
vocab_size = 8
embed_dim = 4
embedding = nn.Embedding(
    num_embeddings = vocab_size,
    embedding_dim = embed_dim
)
print("Before customer: ", embedding.weight)
init_weights = torch.tensor([
    [-0.1882,  0.5530,  1.6267,  0.7013],
    [ 1.7840, -0.8278, -0.2701,  1.3586],
    [ 1.0281, -1.9094,  0.3182,  0.4211],
    [-1.3083, -0.0987,  0.7647, -0.3680],
    [ 0.2293,  1.3255,  0.1318,  2.0501],
    [0.4058, -0.6624, -0.8745,  0.7203],
    [ 0.5582,  0.0786, -0.6817,  0.6902],
    [0.4309, -1.3067, -0.8823,  1.5977]
], dtype = torch.float32)
embedding.weight = nn.Parameter(init_weights)
print('After customer: ', embedding.weight)

Before customer:  Parameter containing:
tensor([[-0.1117,  0.6926, -0.4619, -0.6593],
        [ 0.1236,  1.4529, -2.0941, -0.8224],
        [ 2.1429,  0.2174,  0.0237,  0.2902],
        [-0.1918, -0.7947, -0.3142, -0.7042],
        [-1.1792, -1.4704, -0.0696, -0.0057],
        [-0.1973, -1.9762,  0.9634,  0.1052],
        [-1.0920, -0.9429,  0.3422,  0.0338],
        [ 0.2426,  0.4464, -0.8951, -0.2752]], requires_grad=True)
After customer:  Parameter containing:
tensor([[-0.1882,  0.5530,  1.6267,  0.7013],
        [ 1.7840, -0.8278, -0.2701,  1.3586],
        [ 1.0281, -1.9094,  0.3182,  0.4211],
        [-1.3083, -0.0987,  0.7647, -0.3680],
        [ 0.2293,  1.3255,  0.1318,  2.0501],
        [ 0.4058, -0.6624, -0.8745,  0.7203],
        [ 0.5582,  0.0786, -0.6817,  0.6902],
        [ 0.4309, -1.3067, -0.8823,  1.5977]], requires_grad=True)


In [19]:
input = torch.tensor([0, 4, 7, 2, 1], dtype = torch.long)
label = torch.tensor([1], dtype = torch.float32)

x = embedding(input)
x = nn.Flatten(0)(x)
x = nn.Linear(20, 1)(x)
output = nn.Sigmoid()(x)

In [21]:
loss = nn.BCELoss()(output, label)
optimizer = torch.optim.Adam(embedding.parameters(), lr = 1e-3)
optimizer.zero_grad()

loss.backward()
optimizer.step()
print(f'After update parameters: {embedding.weight}')

After update parameters: Parameter containing:
tensor([[-0.1872,  0.5540,  1.6257,  0.7023],
        [ 1.7830, -0.8268, -0.2711,  1.3576],
        [ 1.0291, -1.9084,  0.3192,  0.4201],
        [-1.3083, -0.0987,  0.7647, -0.3680],
        [ 0.2283,  1.3265,  0.1308,  2.0491],
        [ 0.4058, -0.6624, -0.8745,  0.7203],
        [ 0.5582,  0.0786, -0.6817,  0.6902],
        [ 0.4319, -1.3057, -0.8833,  1.5987]], requires_grad=True)
